In [1]:
# Imports
import sys
from pathlib import Path

# Para que el notebook vea src/ desde notebooks/
sys.path.insert(0, str(Path("..").resolve()))

from src.storage.sqlite_storage import SQLiteStorage
from src.analysis import transformations as tx

# Cargar las tres series
storage = SQLiteStorage("../data/bcch.db")

df_ipc = storage.load_series("ipc")
df_tpm = storage.load_series("tpm")

""" Se introduce una correccion para el caso de USD/CLP, que tiene datos diarios pero con muchos días sin tipo de cambio (fines de semana, feriados). Se detecto
dentro al ejecutar el notebook que el cálculo de la volatilidad (rolling std) estaba dando valores muy bajos, lo que se debía a que se estaban incluyendo los días
 sin tipo de cambio (valor NaN) en el cálculo. 
Al eliminar esos días, la volatilidad refleja mejor las fluctuaciones reales del tipo de cambio.

"""
df_usd = storage.load_series("usd_clp")
df_usd = (df_usd
          .sort_values("fecha")
          .dropna(subset=["valor"])           # solo días con tipo de cambio
          .reset_index(drop=True))

df_usd["ret_1d"]  = tx.pct_change(df_usd, periods=1)
df_usd["ma_30d"]  = tx.rolling_mean(df_usd, window=30)
df_usd["vol_30d"] = tx.rolling_std(df_usd, window=30, value_col="ret_1d")


print(f"IPC:     {len(df_ipc):>5} obs   {df_ipc['fecha'].min().date()} → {df_ipc['fecha'].max().date()}")
print(f"TPM:     {len(df_tpm):>5} obs   {df_tpm['fecha'].min().date()} → {df_tpm['fecha'].max().date()}")
print(f"USD/CLP: {len(df_usd):>5} obs   {df_usd['fecha'].min().date()} → {df_usd['fecha'].max().date()}")

IPC:       197 obs   2009-12-01 → 2026-04-01
TPM:      6009 obs   2009-12-01 → 2026-05-14
USD/CLP:  4096 obs   2009-12-01 → 2026-05-14


In [2]:
df_ipc["mom"]   = tx.pct_change(df_ipc, periods=1)    # variación mensual %
df_ipc["yoy"]   = tx.pct_change(df_ipc, periods=12)   # variación interanual %
df_ipc["ma_3m"] = tx.rolling_mean(df_ipc, window=3, value_col="mom")  # IPC subyacente "casero"

df_ipc.tail(15)


,fecha,valor,mom,yoy,ma_3m
182,2025-02-01,107.16,0.393479,4.730258,0.418484
183,2025-03-01,107.70,0.503919,4.868549,0.652601
184,2025-04-01,107.91,0.194986,4.523441,0.364128
185,2025-05-01,108.12,0.194607,4.443586,0.297837
186,2025-06-01,107.68,-0.406955,4.119126,-0.005788
187,2025-07-01,108.62,0.872957,4.251848,0.220203
188,2025-08-01,108.66,0.036826,4.030637,0.167609
189,2025-09-01,109.14,0.441745,4.400230,0.450509
190,2025-10-01,109.19,0.045813,3.438803,0.174794
191,2025-11-01,109.47,0.256434,3.439478,0.247997


In [3]:
df_tpm["diff_pp"] = tx.diff(df_tpm, periods=1)

# Detectar reuniones de política monetaria: días en que la TPM efectivamente cambió
cambios = df_tpm[df_tpm["diff_pp"].abs() > 1e-9].copy()
print(f"Cambios de TPM detectados: {len(cambios)}")
cambios[["fecha", "valor", "diff_pp"]].tail(10)

Cambios de TPM detectados: 50


,fecha,valor,diff_pp
5132,2023-12-20,8.25,-0.75
5175,2024-02-01,7.25,-1.00
5237,2024-04-03,6.50,-0.75
5288,2024-05-24,6.00,-0.50
5314,2024-06-19,5.75,-0.25
5391,2024-09-04,5.50,-0.25
5435,2024-10-18,5.25,-0.25
5496,2024-12-18,5.00,-0.25
5720,2025-07-30,4.75,-0.25
5860,2025-12-17,4.50,-0.25


In [4]:
df_usd["ret_1d"]  = tx.pct_change(df_usd, periods=1)
df_usd["vol_30d"] = tx.rolling_std(df_usd, window=30, value_col="ret_1d")
df_usd["ma_30d"]  = tx.rolling_mean(df_usd, window=30)

df_usd.tail(10)

,fecha,valor,ret_1d,ma_30d,vol_30d
4086,2026-04-30,901.76,0.639488,904.722333,0.877306
4087,2026-05-04,903.05,0.143054,904.410333,0.867813
4088,2026-05-05,910.01,0.770721,904.060000,0.863755
4089,2026-05-06,905.36,-0.510983,903.449667,0.864965
4090,2026-05-07,892.83,-1.383980,902.705667,0.884659
4091,2026-05-08,887.71,-0.573457,901.770667,0.888748
4092,2026-05-11,890.89,0.358225,901.006000,0.892362
4093,2026-05-12,894.25,0.377151,900.039333,0.872031
4094,2026-05-13,899.91,0.632933,899.039667,0.869331
4095,2026-05-14,889.19,-1.191230,897.627000,0.889617


In [5]:
# USD/CLP: dos variantes interesantes
df_usd_cierre = tx.to_monthly(df_usd[["fecha", "valor"]], method="last")
df_usd_prom   = tx.to_monthly(df_usd[["fecha", "valor"]], method="mean")

# TPM: el "cierre" es lo natural (qué nivel regía a fin de mes)
df_tpm_mes = tx.to_monthly(df_tpm[["fecha", "valor"]], method="last")

print(f"USD/CLP cierre mensual: {len(df_usd_cierre)} filas")
print(f"TPM mensual:            {len(df_tpm_mes)} filas")
df_usd_cierre.tail(5)

USD/CLP cierre mensual: 198 filas
TPM mensual:            198 filas


,fecha,valor
193,2026-01-01,858.45
194,2026-02-01,861.19
195,2026-03-01,931.57
196,2026-04-01,901.76
197,2026-05-01,889.19


In [6]:
df_wide = tx.merge_wide({
    "ipc":     df_ipc[["fecha", "valor"]],
    "tpm":     df_tpm_mes,
    "usd_clp": df_usd_prom,
})

# Variaciones a nivel del panel mensual
df_wide["ipc_mom"] = df_wide["ipc"].pct_change(1)  * 100
df_wide["ipc_yoy"] = df_wide["ipc"].pct_change(12) * 100
df_wide["usd_yoy"] = df_wide["usd_clp"].pct_change(12) * 100

df_wide.tail(15)

,fecha,ipc,tpm,usd_clp,ipc_mom,ipc_yoy,usd_yoy
183,2025-03-01,107.70,5.00,932.551905,0.503919,4.868549,-3.655275
184,2025-04-01,107.91,5.00,961.957143,0.194986,4.523441,0.189448
185,2025-05-01,108.12,5.00,941.012500,0.194607,4.443586,2.520529
186,2025-06-01,107.68,5.00,938.037000,-0.406955,4.119126,1.291026
187,2025-07-01,108.62,4.75,951.550000,0.872957,4.251848,1.492073
188,2025-08-01,108.66,4.75,966.303500,0.036826,4.030637,3.915308
189,2025-09-01,109.14,4.75,960.367500,0.441745,4.400230,3.687381
190,2025-10-01,109.19,4.75,953.973182,0.045813,3.438803,2.158990
191,2025-11-01,109.47,4.75,935.697500,0.256434,3.439478,-3.695193
192,2025-12-01,109.26,4.50,916.163000,-0.191833,3.446317,-6.732492


In [7]:
cols = ["ipc_yoy", "tpm", "usd_yoy"]
df_wide[cols].corr().round(2)


,ipc_yoy,tpm,usd_yoy
ipc_yoy,1.00,0.69,0.32
tpm,0.69,1.00,0.02
usd_yoy,0.32,0.02,1.00


In [8]:
# Correlaciones con rezago: ¿el cambio de TPM hoy correlaciona con IPC de los próximos meses?
for lag in range(0, 7):
    corr = df_wide["ipc_yoy"].corr(df_wide["tpm"].shift(lag))
    print(f"lag={lag:>2}m  corr(IPC_yoy, TPM_{lag}m_atras) = {corr:.3f}")


lag= 0m  corr(IPC_yoy, TPM_0m_atras) = 0.692
lag= 1m  corr(IPC_yoy, TPM_1m_atras) = 0.649
lag= 2m  corr(IPC_yoy, TPM_2m_atras) = 0.599
lag= 3m  corr(IPC_yoy, TPM_3m_atras) = 0.545
lag= 4m  corr(IPC_yoy, TPM_4m_atras) = 0.485
lag= 5m  corr(IPC_yoy, TPM_5m_atras) = 0.423
lag= 6m  corr(IPC_yoy, TPM_6m_atras) = 0.358


In [9]:
ultimo = df_ipc.dropna(subset=["yoy"]).iloc[-1]
print(f"Mes más reciente con y/y calculado: {ultimo['fecha'].date()}")
print(f"IPC nivel: {ultimo['valor']:.2f}")
print(f"IPC y/y:   {ultimo['yoy']:.2f}%")


Mes más reciente con y/y calculado: 2026-04-01
IPC nivel: 112.18
IPC y/y:   3.96%


In [10]:
# Se cambio la serie de IPC ya que esta se hizo un salto de base de 2018 a 2023 por lo que existe una variacion de -60%, monto totalmente irreal por lo que se decide usar la serie base 2023

from datetime import date
from src.extract.series_extractor import SeriesExtractor
from src.storage.sqlite_storage import SQLiteStorage

# Re-instanciar para que el SeriesExtractor relea el series.yaml actualizado
extractor = SeriesExtractor()
storage = SQLiteStorage()

# Descargar solo la serie corregida
df_ipc_nuevo = extractor.extract_one(
    "ipc",
    start=date(2023, 1, 1),
    end=date.today(),
)
print(f"Descargadas {len(df_ipc_nuevo)} filas para IPC")
df_ipc_nuevo.head()

Descargadas 40 filas para IPC


,fecha,valor,serie,codigo
0,2023-01-01,98.00,ipc,F074.IPC.IND.Z.EP23.C.M
1,2023-02-01,97.93,ipc,F074.IPC.IND.Z.EP23.C.M
2,2023-03-01,99.00,ipc,F074.IPC.IND.Z.EP23.C.M
3,2023-04-01,99.30,ipc,F074.IPC.IND.Z.EP23.C.M
4,2023-05-01,99.41,ipc,F074.IPC.IND.Z.EP23.C.M


In [11]:
print("Primeras filas:")
print(df_ipc_nuevo.head())
print("\nÚltimas filas:")
print(df_ipc_nuevo.tail())
print(f"\nNivel min/max: {df_ipc_nuevo['valor'].min():.3f} / {df_ipc_nuevo['valor'].max():.3f}")

Primeras filas:
       fecha  valor serie                   codigo
0 2023-01-01  98.00   ipc  F074.IPC.IND.Z.EP23.C.M
1 2023-02-01  97.93   ipc  F074.IPC.IND.Z.EP23.C.M
2 2023-03-01  99.00   ipc  F074.IPC.IND.Z.EP23.C.M
3 2023-04-01  99.30   ipc  F074.IPC.IND.Z.EP23.C.M
4 2023-05-01  99.41   ipc  F074.IPC.IND.Z.EP23.C.M

Últimas filas:
        fecha   valor serie                   codigo
35 2025-12-01  109.26   ipc  F074.IPC.IND.Z.EP23.C.M
36 2026-01-01  109.71   ipc  F074.IPC.IND.Z.EP23.C.M
37 2026-02-01  109.70   ipc  F074.IPC.IND.Z.EP23.C.M
38 2026-03-01  110.75   ipc  F074.IPC.IND.Z.EP23.C.M
39 2026-04-01  112.18   ipc  F074.IPC.IND.Z.EP23.C.M

Nivel min/max: 97.930 / 112.180


In [12]:
n = storage.save_observations(df_ipc_nuevo)
print(f"Guardadas/actualizadas {n} filas en SQLite")
print()
print(storage.summary())

Guardadas/actualizadas 40 filas en SQLite

     serie  n_obs primera_fecha ultima_fecha
0      ipc    197    2009-12-01   2026-04-01
1      tpm   6009    2009-12-01   2026-05-14
2  usd_clp   6009    2009-12-01   2026-05-14


In [13]:
df_ipc = storage.load_series("ipc")
df_ipc["mom"] = tx.pct_change(df_ipc, periods=1)
df_ipc["yoy"] = tx.pct_change(df_ipc, periods=12)

print(df_ipc.dropna(subset=["yoy"]).tail(15))

         fecha   valor       mom       yoy
182 2025-02-01  107.16  0.393479  4.730258
183 2025-03-01  107.70  0.503919  4.868549
184 2025-04-01  107.91  0.194986  4.523441
185 2025-05-01  108.12  0.194607  4.443586
186 2025-06-01  107.68 -0.406955  4.119126
187 2025-07-01  108.62  0.872957  4.251848
188 2025-08-01  108.66  0.036826  4.030637
189 2025-09-01  109.14  0.441745  4.400230
190 2025-10-01  109.19  0.045813  3.438803
191 2025-11-01  109.47  0.256434  3.439478
192 2025-12-01  109.26 -0.191833  3.446317
193 2026-01-01  109.71  0.411862  2.782462
194 2026-02-01  109.70 -0.009115  2.370287
195 2026-03-01  110.75  0.957156  2.831941
196 2026-04-01  112.18  1.291196  3.957001


In [14]:
import pandas as pd

df_ipc = storage.load_series("ipc")
df_ipc["mom"] = tx.pct_change(df_ipc, periods=1)
df_ipc["yoy"] = tx.pct_change(df_ipc, periods=12)

# Mostrar los últimos 6 meses con su y/y, m/m y nivel
print(df_ipc.dropna(subset=["yoy"]).tail(6).to_string(index=False))

     fecha  valor       mom      yoy
2025-11-01 109.47  0.256434 3.439478
2025-12-01 109.26 -0.191833 3.446317
2026-01-01 109.71  0.411862 2.782462
2026-02-01 109.70 -0.009115 2.370287
2026-03-01 110.75  0.957156 2.831941
2026-04-01 112.18  1.291196 3.957001


In [15]:
# Extension para considerar que el IPC se cambio de serie a una mas longeva

from datetime import date
from src.extract.series_extractor import SeriesExtractor
from src.storage.sqlite_storage import SQLiteStorage

extractor = SeriesExtractor()
storage = SQLiteStorage()

# Re-extraer TPM y USD/CLP con histórico extendido
for alias in ["tpm", "usd_clp"]:
    df = extractor.extract_one(alias, start=date(2009, 12, 1), end=date.today())
    n = storage.save_observations(df)
    print(f"{alias}: {n} filas insertadas/actualizadas")

print()
print(storage.summary())

tpm: 6009 filas insertadas/actualizadas
usd_clp: 6009 filas insertadas/actualizadas

     serie  n_obs primera_fecha ultima_fecha
0      ipc    197    2009-12-01   2026-04-01
1      tpm   6009    2009-12-01   2026-05-14
2  usd_clp   6009    2009-12-01   2026-05-14


In [16]:
#Carga de las tres series para analisis
from src.analysis import transformations as tx

df_ipc = storage.load_series("ipc")
df_tpm = storage.load_series("tpm")
df_usd = storage.load_series("usd_clp")

print(f"IPC:     {len(df_ipc):>6} obs   {df_ipc['fecha'].min().date()} → {df_ipc['fecha'].max().date()}")
print(f"TPM:     {len(df_tpm):>6} obs   {df_tpm['fecha'].min().date()} → {df_tpm['fecha'].max().date()}")
print(f"USD/CLP: {len(df_usd):>6} obs   {df_usd['fecha'].min().date()} → {df_usd['fecha'].max().date()}")

IPC:        197 obs   2009-12-01 → 2026-04-01
TPM:       6009 obs   2009-12-01 → 2026-05-14
USD/CLP:   6009 obs   2009-12-01 → 2026-05-14


In [17]:
# IPC: variación mensual, interanual y MA suavizada
df_ipc["mom"]   = tx.pct_change(df_ipc, periods=1)
df_ipc["yoy"]   = tx.pct_change(df_ipc, periods=12)
df_ipc["ma_3m"] = tx.rolling_mean(df_ipc, window=3, value_col="mom")

# TPM: diferencia en p.p.
df_tpm["diff_pp"] = tx.diff(df_tpm, periods=1)

# USD/CLP: retorno diario, MA 30d, volatilidad 30d
df_usd["ret_1d"]  = tx.pct_change(df_usd, periods=1)
df_usd["ma_30d"]  = tx.rolling_mean(df_usd, window=30)
df_usd["vol_30d"] = tx.rolling_std(df_usd, window=30, value_col="ret_1d")

print("IPC (últimas 6 filas):")
print(df_ipc.tail(6).to_string(index=False))
print("\nTPM (cambios detectados, últimos 5):")
cambios = df_tpm[df_tpm["diff_pp"].abs() > 1e-9]
print(cambios.tail(5).to_string(index=False))
print("\nUSD/CLP (últimas 5 filas):")
print(df_usd.tail(5).to_string(index=False))

IPC (últimas 6 filas):
     fecha  valor       mom      yoy    ma_3m
2025-11-01 109.47  0.256434 3.439478 0.247997
2025-12-01 109.26 -0.191833 3.446317 0.036804
2026-01-01 109.71  0.411862 2.782462 0.158821
2026-02-01 109.70 -0.009115 2.370287 0.070304
2026-03-01 110.75  0.957156 2.831941 0.453301
2026-04-01 112.18  1.291196 3.957001 0.746412

TPM (cambios detectados, últimos 5):
     fecha  valor  diff_pp
2024-09-04   5.50    -0.25
2024-10-18   5.25    -0.25
2024-12-18   5.00    -0.25
2025-07-30   4.75    -0.25
2025-12-17   4.50    -0.25

USD/CLP (últimas 5 filas):
     fecha  valor    ret_1d  ma_30d  vol_30d
2026-05-10    NaN       NaN     NaN      NaN
2026-05-11 890.89       NaN     NaN      NaN
2026-05-12 894.25  0.377151     NaN      NaN
2026-05-13 899.91  0.632933     NaN      NaN
2026-05-14 889.19 -1.191230     NaN      NaN


In [18]:
df_tpm_mes = tx.to_monthly(df_tpm[["fecha", "valor"]], method="last")
df_usd_mes = tx.to_monthly(df_usd[["fecha", "valor"]], method="mean")

print(f"TPM mensual: {len(df_tpm_mes)} filas")
print(f"USD/CLP mensual: {len(df_usd_mes)} filas")
df_usd_mes.tail(3)

TPM mensual: 198 filas
USD/CLP mensual: 198 filas


,fecha,valor
195,2026-03-01,909.893182
196,2026-04-01,897.886190
197,2026-05-01,897.022222


In [19]:
df_wide = tx.merge_wide({
    "ipc":     df_ipc[["fecha", "valor"]],
    "tpm":     df_tpm_mes,
    "usd_clp": df_usd_mes,
})

# Variaciones a nivel panel
df_wide["ipc_mom"] = df_wide["ipc"].pct_change(1)  * 100
df_wide["ipc_yoy"] = df_wide["ipc"].pct_change(12) * 100
df_wide["usd_yoy"] = df_wide["usd_clp"].pct_change(12) * 100

print(f"Panel mensual: {len(df_wide)} filas, {df_wide['fecha'].min().date()} → {df_wide['fecha'].max().date()}")
print()
print(df_wide.tail(10).to_string(index=False))

Panel mensual: 198 filas, 2009-12-01 → 2026-05-01

     fecha    ipc  tpm    usd_clp   ipc_mom  ipc_yoy    usd_yoy
2025-08-01 108.66 4.75 966.303500  0.036826 4.030637   3.915308
2025-09-01 109.14 4.75 960.367500  0.441745 4.400230   3.687381
2025-10-01 109.19 4.75 953.973182  0.045813 3.438803   2.158990
2025-11-01 109.47 4.75 935.697500  0.256434 3.439478  -3.695193
2025-12-01 109.26 4.50 916.163000 -0.191833 3.446317  -6.732492
2026-01-01 109.71 4.50 883.965714  0.411862 2.782462 -11.670880
2026-02-01 109.70 4.50 862.020000 -0.009115 2.370287  -9.888984
2026-03-01 110.75 4.50 909.893182  0.957156 2.831941  -2.429755
2026-04-01 112.18 4.50 897.886190  1.291196 3.957001  -6.660479
2026-05-01    NaN 4.50 897.022222       NaN      NaN  -4.674781


In [20]:
 #Contemporáneas
cols = ["ipc_yoy", "tpm", "usd_yoy"]
print("Correlaciones contemporáneas:")
print(df_wide[cols].corr().round(2))

# Con rezago: ¿IPC reacciona a TPM con cuántos meses?
print("\nCorr(IPC_yoy, TPM rezagada N meses):")
for lag in range(0, 13):
    c = df_wide["ipc_yoy"].corr(df_wide["tpm"].shift(lag))
    print(f"  lag={lag:>2}m  r={c:.3f}")

# ¿USD anticipa IPC?
print("\nCorr(IPC_yoy, USD_yoy rezagado N meses):")
for lag in range(0, 13):
    c = df_wide["ipc_yoy"].corr(df_wide["usd_yoy"].shift(lag))
    print(f"  lag={lag:>2}m  r={c:.3f}")

Correlaciones contemporáneas:
         ipc_yoy   tpm  usd_yoy
ipc_yoy     1.00  0.69     0.32
tpm         0.69  1.00     0.02
usd_yoy     0.32  0.02     1.00

Corr(IPC_yoy, TPM rezagada N meses):
  lag= 0m  r=0.692
  lag= 1m  r=0.649
  lag= 2m  r=0.599
  lag= 3m  r=0.545
  lag= 4m  r=0.485
  lag= 5m  r=0.423
  lag= 6m  r=0.358
  lag= 7m  r=0.290
  lag= 8m  r=0.224
  lag= 9m  r=0.158
  lag=10m  r=0.095
  lag=11m  r=0.037
  lag=12m  r=-0.017

Corr(IPC_yoy, USD_yoy rezagado N meses):
  lag= 0m  r=0.324
  lag= 1m  r=0.352
  lag= 2m  r=0.361
  lag= 3m  r=0.356
  lag= 4m  r=0.351
  lag= 5m  r=0.335
  lag= 6m  r=0.311
  lag= 7m  r=0.279
  lag= 8m  r=0.241
  lag= 9m  r=0.193
  lag=10m  r=0.144
  lag=11m  r=0.090
  lag=12m  r=0.033


In [21]:
from pathlib import Path

out_dir = Path("../data/processed")
out_dir.mkdir(exist_ok=True)

df_wide.to_parquet(out_dir / "panel_mensual.parquet", index=False)
df_ipc.to_parquet(out_dir / "ipc_enriquecido.parquet", index=False)
df_usd.to_parquet(out_dir / "usd_enriquecido.parquet", index=False)
df_tpm.to_parquet(out_dir / "tpm_enriquecido.parquet", index=False)

print("Procesados escritos en data/processed/")

Procesados escritos en data/processed/
